# Тема 19. RAG (Retrieval-Augmented Generation)

**Домен:** информационная безопасность (корпус документов PDF/текст).  
**Автор:** Гладков Роман Евгеньевич, **Группа:** КТсо3-5 

**Аннотация:** Реализован пайплайн PDF/текст → нормализация → чанки → эмбеддинги BERTA → FAISS → извлечение (MMR/top-k) → генерация Qwen3-0.6B на CPU с JSON-ответом и цитатами.  
Сравниваются режимы **без RAG** и **с RAG** на ≥10 вопросах; сохраняются таблицы документов и чанков, trace, график прокси groundedness, KPI-отчёт и архитектурные артефакты в `artifacts/`.


## 1. Цель, входы/выходы, критерии приёмки

**Цель:** чат-бот (однопроходная генерация по вопросу), отвечающий по корпусу с опорой на извлечённый контекст; ответы структурированы JSON (поля `answer`, `citations`, `confidence`, `notes`) по схеме `starter_pack/schemas/llm_answer.schema.json`.

**Входы (`starter_pack/`):**
- `docs/corpus/` — ≥20 документов (основной корпус в `starter_pack/docs/corpus/`);
- `docs/domain_brief.md`, `docs/sources_index.csv` (обновляется из метаданных);
- `tests/questions.csv` — вопросы для оценки;
- `prompts/prompt_templates.md`.

**Выходы (`artifacts/`):** `input_profile.json`, `documents.csv`, `chunks.csv`, `results.csv`, `trace.jsonl`, `fig_01.png`, `architecture.mmd`, `architecture.svg`, `kpi_report.csv`, `export_summary.txt`.

### KPI (заполнение факта — в ячейке `evaluate()`)

| KPI | Определение | Порог | Где измеряем | Факт |
|-----|-------------|-------|--------------|------|
| kpi_corpus | Число файлов в корпусе | ≥20 | валидация входов | |
| kpi_questions | Число вопросов в эксперименте | ≥10 | questions.csv / results | |
| kpi_pipeline | Индекс FAISS построен | да/нет | build_vectorstore | |
| kpi_citations | Доля ответов RAG с непустыми citations | ≥0.8 | results | |
| kpi_grounded_proxy | Средняя эвристическая groundedness | ≥0.8 | results | |

**Acceptance criteria (Given/When/Then):**
1. **Given** корпус из ≥20 файлов в `starter_pack/docs/corpus/`, **when** выполняется раздел индексации, **then** FAISS-индекс создаётся без ошибок.
2. **Given** список вопросов из `questions.csv`, **when** прогон baseline и RAG, **then** в `artifacts/results.csv` есть столбцы no_rag / rag и citations.
3. **Given** RAG-режим, **when** модель отвечает, **then** JSON содержит ссылки на файлы источников из корпуса.
4. **Given** финальный экспорт, **then** в `artifacts/` присутствуют файлы из контракта шаблона.


## 2. Среда выполнения и воспроизводимость


In [1]:
import importlib
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

for pkg in ("langchain", "langchain_core", "langchain_community", "sentence_transformers", "transformers", "faiss"):
    try:
        m = importlib.import_module(pkg)
        print(pkg, getattr(m, "__version__", "?"))
    except Exception as e:
        print(pkg, "ERR", e)

print("torch", torch.__version__)
print("python", sys.version)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "starter_pack").is_dir():
    raise RuntimeError("Запускайте ноутбук из каталога Тема19_rag (где лежит starter_pack).")

CONFIG = {
    "seed": 42,
    "project_root": str(PROJECT_ROOT),
    "corpus_dir": "starter_pack/docs/corpus",
    "artifacts_path": "artifacts/",
    "questions_path": "starter_pack/tests/questions.csv",
    "sources_index_path": "starter_pack/docs/sources_index.csv",
    "domain_brief_path": "starter_pack/docs/domain_brief.md",
    "schema_path": "starter_pack/schemas/llm_answer.schema.json",
    "embedding_model": "sergeyzh/BERTA",
    "llm_model": "Qwen/Qwen3-0.6B",
    "chunk_size": 900,
    "chunk_overlap": 120,
    "retrieval_k": 5,
    "retrieval_fetch_k": 20,
    "use_mmr": True,
    "llm_max_new_tokens": 384,
    "llm_temperature": 0.7,
    "llm_top_p": 0.8,
}

os.environ["TOKENIZERS_PARALLELISM"] = "false"
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print("CONFIG keys:", sorted(CONFIG.keys()))


langchain 1.2.17
langchain_core 1.3.2
langchain_community 0.4.1


e:\Work\Тема19_rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence_transformers 5.4.1
transformers 5.7.0
faiss 1.13.2
torch 2.11.0+cpu
python 3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]
CONFIG keys: ['artifacts_path', 'chunk_overlap', 'chunk_size', 'corpus_dir', 'domain_brief_path', 'embedding_model', 'llm_max_new_tokens', 'llm_model', 'llm_temperature', 'llm_top_p', 'project_root', 'questions_path', 'retrieval_fetch_k', 'retrieval_k', 'schema_path', 'seed', 'sources_index_path', 'use_mmr']


## 3. Загрузка starter-pack и первичная валидация входов


In [2]:
from pathlib import Path
import json

import pandas as pd

corpus_dir = Path(CONFIG["project_root"]) / CONFIG["corpus_dir"]
files = [p for p in corpus_dir.iterdir() if p.is_file() and p.suffix.lower() in {".pdf", ".txt", ".md"} and p.name.lower() != "readme.md"]
assert len(files) >= 20, f"В {corpus_dir} должно быть ≥20 файлов .pdf/.txt/.md (без README.md). Сейчас: {len(files)}"

qpath = Path(CONFIG["project_root"]) / CONFIG["questions_path"]
questions_df = pd.read_csv(qpath)
assert "question" in questions_df.columns
assert len(questions_df) >= 10, "Нужно ≥10 вопросов в questions.csv"

spath = Path(CONFIG["project_root"]) / CONFIG["sources_index_path"]
if spath.exists():
    src_idx = pd.read_csv(spath)
    for col in ("doc_id", "title", "source", "size_bytes"):
        assert col in src_idx.columns, f"sources_index.csv: нет колонки {col}"

profile = {
    "corpus_files": len(files),
    "questions_rows": int(len(questions_df)),
    "corpus_dir": str(corpus_dir),
    "filenames": [p.name for p in sorted(files)][:50],
}
Path(CONFIG["artifacts_path"]).mkdir(parents=True, exist_ok=True)
with open(Path(CONFIG["artifacts_path"]) / "input_profile.json", "w", encoding="utf-8") as f:
    json.dump(profile, f, ensure_ascii=False, indent=2)
print(json.dumps(profile, ensure_ascii=False, indent=2)[:2000])


{
  "corpus_files": 20,
  "questions_rows": 12,
  "corpus_dir": "E:\\Work\\Тема19_rag\\starter_pack\\docs\\corpus",
  "filenames": [
    "001_Карцан_Современные+кибератаки+классификация+и+способы+защиты+(1).pdf",
    "14_p130.pdf",
    "172-175.pdf",
    "2083.dokl.pdf",
    "978-5-400-01865-7_2025_342_346.pdf",
    "Cyber.pdf",
    "digital-tech-security_1_112_35445.pdf",
    "gorev.pdf",
    "HSE CS MC Klimov 14 мая.pdf",
    "Informatsionnaya_bezopasnost_novaya.pdf",
    "kiberucheniya.pdf",
    "konspekt-lektsij-po-IB.pdf",
    "Makarenko-ib.pdf",
    "Metodicheskie-rekomendatsii-po-razrabotke-Plana-reagirovaniya.pdf",
    "Информационная-безопасность-и-управление-1.pdf",
    "Карточки по Кибербезопасности.pdf",
    "Лекция 12. Вредоносные программы и способы защиты от них.pdf",
    "Михнев.pdf",
    "СТ_лекция14.pdf",
    "Что такое информационная безопасность.pdf"
  ]
}


## 4. Теоретическая часть

**Тезисы:**
1. RAG снижает галлюцинации за счёт ограничения знания извлечённым контекстом (при корректном retrieval).
2. Качество зависит от стратегии чанкинга, размера overlap и выбора эмбеддингов; для асимметричных моделей важны разные префиксы запроса и документа.
3. FAISS обеспечивает быстрый приближённый поиск по плотным векторам; MMR помогает разнообразить top-k.
4. На CPU малые LLM (Qwen3-0.6B) дают приемлемую скорость для учебного прототипа при ограниченной длине генерации.

**Источники (использовались при выборе стека):**
- [LangChain documentation](https://python.langchain.com/docs/)
- [FAISS](https://github.com/facebookresearch/faiss)
- [BERTA model card](https://huggingface.co/sergeyzh/BERTA)
- [Qwen3-0.6B model card](https://huggingface.co/Qwen/Qwen3-0.6B)

**Ограничения:**
1. Эвристическая groundedness — прокси; при наличии колонки `groundedness_human` в `questions.csv` её удобнее использовать для сверки.
2. CPU-инференс ограничивает длину контекста и число экспериментов за разумное время.
3. PDF с таблицами и сканами без OCR могут давать шумный текст.


## 5. Проектирование (Design)

### Архитектура (Mermaid)

```mermaid
flowchart LR
  subgraph ingest[Загрузка]
    D[PDF/TXT/MD] --> L[LangChain loaders]
    L --> S[RecursiveCharacterTextSplitter]
  end
  subgraph index[Индекс]
    S --> E[BERTA embeddings]
    E --> F[FAISS VectorStore]
  end
  subgraph query[Запрос]
    Q[Вопрос] --> R[Retriever MMR/top-k]
    F --> R
    R --> C[Контекст + цитаты]
    C --> G[Qwen3-0.6B CPU]
    G --> J[JSON ответ]
  end
```

### Контракт данных

| Сущность | Поля |
|----------|------|
| Документ | `source` (имя файла), `page_content` |
| Чанк | `chunk_id`, `doc_id`, `text`, `length` |
| Ответ | `answer`, `citations[]`, `confidence`, `notes` |

**Допущения:** документы на русском/английском; эмбеддинги BERTA с префиксами `search_document:` / `search_query:`; Qwen с `enable_thinking=False` через chat template.


## 6. Реализация (build / run)


In [ ]:
from pathlib import Path

import pandas as pd

from rag_lib import (
    BertaEmbeddings,
    build_documents_table_from_chunks,
    build_qwen_llm,
    build_vectorstore,
    chunks_table,
    heuristic_grounded,
    load_corpus_documents,
    no_rag_answer,
    rag_answer,
    set_seed,
    split_documents,
)

set_seed(CONFIG["seed"])

corpus_dir = Path(CONFIG["project_root"]) / CONFIG["corpus_dir"]
raw_docs = load_corpus_documents(corpus_dir)
assert raw_docs, "Не удалось загрузить документы (пустой корпус после loaders)."

chunks = split_documents(
    raw_docs,
    chunk_size=int(CONFIG["chunk_size"]),
    chunk_overlap=int(CONFIG["chunk_overlap"]),
)
print("chunks:", len(chunks))

embeddings = BertaEmbeddings(CONFIG["embedding_model"], device="cpu")
vectorstore = build_vectorstore(chunks, embeddings)

llm, tokenizer = build_qwen_llm(
    CONFIG["llm_model"],
    max_new_tokens=int(CONFIG["llm_max_new_tokens"]),
    temperature=float(CONFIG["llm_temperature"]),
    top_p=float(CONFIG["llm_top_p"]),
)

SYSTEM_RAG = '''Ты аналитик по внутренним документам по информационной безопасности.
Отвечай только на основе контекста ниже. Если в контексте нет ответа — напиши об этом в поле notes и дай осторожный ответ в answer.
Контекст:
{context}
'''

USER_JSON = (
    "Ответ строго одним JSON-объектом (без markdown-обёртки) с полями: "
    "answer (string), citations (array объектов с полями file, excerpt), "
    "confidence (число 0..1), notes (string). "
    "Поле file в citations должно совпадать с именем файла из контекста."
)

SYSTEM_NO_RAG = '''Ты языковая модель без доступа к внутренним документам организации.
Отвечай из общих знаний, отмечай неопределённость. Формат — JSON как в инструкции пользователя.'''

print("build_* готово: vectorstore + llm")


Overwriting cache for 0 819


chunks: 2860


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8103.48it/s]
Default prompt name is set to 'Classification'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.
Batches:  56%|█████▌    | 50/90 [02:49<02:22,  3.57s/it]

## 7. Эксперименты: baseline (no RAG) vs RAG


In [ ]:
from pathlib import Path
import json

import pandas as pd

from rag_lib import heuristic_grounded

qdf = pd.read_csv(Path(CONFIG["project_root"]) / CONFIG["questions_path"])
rows = []
trace = []

for _, row in qdf.iterrows():
    qid = row["question_id"]
    question = row["question"]
    cat = row.get("category", "")

    out_no = no_rag_answer(llm, tokenizer, question, SYSTEM_NO_RAG, USER_JSON)
    out_rag = rag_answer(
        vectorstore,
        llm,
        tokenizer,
        question,
        SYSTEM_RAG,
        USER_JSON,
        k=int(CONFIG["retrieval_k"]),
        fetch_k=int(CONFIG["retrieval_fetch_k"]),
        use_mmr=bool(CONFIG["use_mmr"]),
    )

    cits = out_rag.get("citations") or []
    retrieved = out_rag.get("_retrieved_sources") or []
    g_proxy = heuristic_grounded(str(out_rag.get("answer", "")), cits, retrieved)

    human = row.get("groundedness_human")
    if pd.isna(human) or human == "":
        g_final = float(g_proxy)
    else:
        g_final = float(human)

    rows.append(
        {
            "question_id": qid,
            "category": cat,
            "question": question,
            "no_rag_answer": (out_no.get("answer") or "")[:4000],
            "rag_answer": (out_rag.get("answer") or "")[:4000],
            "rag_citations_json": json.dumps(cits, ensure_ascii=False),
            "rag_confidence": out_rag.get("confidence"),
            "retrieved_sources": "|".join(retrieved),
            "groundedness_proxy": g_proxy,
            "groundedness_for_kpi": g_final,
        }
    )

    trace.append(
        {
            "question_id": qid,
            "no_rag_raw_tail": (out_no.get("_raw") or "")[-500:],
            "rag_raw_tail": (out_rag.get("_raw") or "")[-500:],
        }
    )

results_df = pd.DataFrame(rows)
Path(CONFIG["artifacts_path"]).mkdir(parents=True, exist_ok=True)
results_df.to_csv(Path(CONFIG["artifacts_path"]) / "results.csv", index=False)

with open(Path(CONFIG["artifacts_path"]) / "trace.jsonl", "w", encoding="utf-8") as f:
    for t in trace:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

print(results_df[["question_id", "groundedness_proxy", "groundedness_for_kpi"]].head(12))


## 8. Оценка KPI (`evaluate`)


In [ ]:
import pandas as pd
from pathlib import Path


def evaluate() -> pd.DataFrame:
    root = Path(CONFIG["project_root"])
    corpus_dir = root / CONFIG["corpus_dir"]
    n_docs = len([p for p in corpus_dir.iterdir() if p.is_file() and p.suffix.lower() in {".pdf", ".txt", ".md"} and p.name.lower() != "readme.md"])

    results_path = root / CONFIG["artifacts_path"] / "results.csv"
    res = pd.read_csv(results_path) if results_path.exists() else pd.DataFrame()

    n_q = len(res) if len(res) else 0
    vs_ok = "vectorstore" in globals() and vectorstore is not None

    if len(res):
        cits_nonempty = res["rag_citations_json"].fillna("[]").map(lambda s: s.strip() not in ("[]", ""))
        cit_rate = float(cits_nonempty.mean())
        gr = float(res["groundedness_for_kpi"].astype(float).mean())
    else:
        cit_rate = 0.0
        gr = 0.0

    rows = [
        {
            "kpi": "kpi_corpus",
            "target": ">=20",
            "actual": str(n_docs),
            "pass": n_docs >= 20,
            "notes": str(corpus_dir),
        },
        {
            "kpi": "kpi_questions",
            "target": ">=10",
            "actual": str(n_q),
            "pass": n_q >= 10,
            "notes": "results.csv rows",
        },
        {
            "kpi": "kpi_pipeline_faiss",
            "target": "built",
            "actual": "yes" if vs_ok else "no",
            "pass": bool(vs_ok),
            "notes": "FAISS from LangChain",
        },
        {
            "kpi": "kpi_citations_rate",
            "target": ">=0.8",
            "actual": f"{cit_rate:.3f}",
            "pass": cit_rate >= 0.8,
            "notes": "доля ответов с непустыми citations",
        },
        {
            "kpi": "kpi_groundedness_mean",
            "target": ">=0.8",
            "actual": f"{gr:.3f}",
            "pass": gr >= 0.8,
            "notes": "mean groundedness_for_kpi (human или proxy)",
        },
    ]
    return pd.DataFrame(rows)


kpi_df = evaluate()
print(kpi_df)
kpi_df.to_csv(Path(CONFIG["artifacts_path"]) / "kpi_report.csv", index=False)


## 9. Анализ, выводы и ограничения

- **Что получилось:** end-to-end пайплайн на LangChain + FAISS + BERTA + Qwen3 на CPU; сравнение no-RAG/RAG; артефакты в `artifacts/`.
- **Что может не хватать:** качество ответов ограничено размером модели и длиной контекста; прокси groundedness грубый.
- **+2 недели:** BM25+векторный гибрид (`rank_bm25`), reranker, подбор chunk_size, квантизация LLM.
- **Риски:** утечка чувствительных данных в логах; неверные цитаты при шумном PDF; долгий прогон на CPU.
- **Меры:** не включать в корпус конфиденциальные данные при открытой публикации репозитория; проверка полей `citations`; кэширование индекса (`vectorstore.save_local`).


## 10. Автоматические проверки (Self-check)


In [ ]:
from pathlib import Path

corpus_dir = Path(CONFIG["project_root"]) / CONFIG["corpus_dir"]
n_docs = len([p for p in corpus_dir.iterdir() if p.is_file() and p.suffix.lower() in {".pdf", ".txt", ".md"} and p.name.lower() != "readme.md"])
assert n_docs >= 20

res_path = Path(CONFIG["project_root"]) / CONFIG["artifacts_path"] / "results.csv"
assert res_path.exists()
import pandas as pd
res = pd.read_csv(res_path)
assert len(res) >= 10
assert set(["no_rag_answer", "rag_answer", "rag_citations_json"]).issubset(res.columns)

# JSON-ответы: поле rag_citations_json парсится
for s in res["rag_citations_json"].head(3):
    import json
    json.loads(s)

print("self-check: OK")


## 11. Экспорт артефактов (таблицы, график, диаграмма, summary)


In [ ]:
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from rag_lib import build_documents_table_from_chunks, chunks_table

art = Path(CONFIG["project_root"]) / CONFIG["artifacts_path"]
art.mkdir(parents=True, exist_ok=True)

docs_df = build_documents_table_from_chunks(Path(CONFIG["project_root"]) / CONFIG["corpus_dir"], chunks)
docs_df.to_csv(art / "documents.csv", index=False)
docs_df.to_csv(Path(CONFIG["project_root"]) / CONFIG["sources_index_path"], index=False)

chunks_df = chunks_table(chunks)
chunks_df.to_csv(art / "chunks.csv", index=False)

shutil.copy2(
    Path(CONFIG["project_root"]) / "starter_pack" / "docs" / "architecture.mmd",
    art / "architecture.mmd",
)

svg = '''<svg xmlns="http://www.w3.org/2000/svg" width="880" height="220">
  <rect width="100%" height="100%" fill="white"/>
  <text x="20" y="40" font-size="18" font-family="Arial">Тема 19 RAG (CPU)</text>
  <rect x="20" y="60" width="150" height="50" fill="#e3f2fd" stroke="#333"/>
  <text x="45" y="90" font-size="12">Corpus</text>
  <rect x="200" y="60" width="150" height="50" fill="#e8f5e9" stroke="#333"/>
  <text x="235" y="90" font-size="12">Chunks</text>
  <rect x="380" y="60" width="150" height="50" fill="#fff3e0" stroke="#333"/>
  <text x="415" y="90" font-size="12">BERTA</text>
  <rect x="560" y="60" width="150" height="50" fill="#fce4ec" stroke="#333"/>
  <text x="610" y="90" font-size="12">FAISS</text>
  <rect x="740" y="60" width="120" height="50" fill="#f3e5f5" stroke="#333"/>
  <text x="760" y="90" font-size="12">Qwen3</text>
  <line x1="170" y1="85" x2="200" y2="85" stroke="#333"/>
  <line x1="350" y1="85" x2="380" y2="85" stroke="#333"/>
  <line x1="530" y1="85" x2="560" y2="85" stroke="#333"/>
  <line x1="710" y1="85" x2="740" y2="85" stroke="#333"/>
</svg>'''
(art / "architecture.svg").write_text(svg, encoding="utf-8")

res = pd.read_csv(art / "results.csv")
plt.figure(figsize=(10, 4))
plt.bar(res["question_id"].astype(str), res["groundedness_for_kpi"].astype(float))
plt.xticks(rotation=45, ha="right")
plt.ylabel("groundedness")
plt.title("Прокси/разметка groundedness по вопросам")
plt.tight_layout()
plt.savefig(art / "fig_01.png", dpi=150)
plt.close()

kpi_df = evaluate()
kpi_df.to_csv(art / "kpi_report.csv", index=False)

files = sorted([p.name for p in art.iterdir() if p.is_file()])
(art / "export_summary.txt").write_text(
    "Сохранено:\n" + "\n".join(files),
    encoding="utf-8",
)
print("export_summary.txt:\n", (art / "export_summary.txt").read_text(encoding="utf-8"))


## 12. Чек-лист готовности артефактов

- [ ] `artifacts/kpi_report.csv` есть и все KPI PASS (при необходимости учитывается колонка `groundedness_human` в `questions.csv`)
- [ ] `artifacts/results.csv` есть
- [ ] Архитектура: `artifacts/architecture.mmd` и `artifacts/architecture.svg`
- [ ] В ноутбуке нет «ручных» путей вне `CONFIG`/`project_root`
- [ ] **Run All** проходит без ошибок на машине
